# Ask Alfonso · training on a GPU (v3)

Trains the from-scratch transformer on Google Colab's free GPU, then evaluates the full
pipeline (retrieval + model + guardrails) on the dev set.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.
Then run the cells in order (`Shift + Enter`). v3 has ~1.6× more training data than v2
plus a pretraining stage, so expect it to take noticeably longer than v2 did. Keep the tab open.

Everything is rebuilt from the GitHub repo, so the model always trains on the latest `src/data`.

**What's new in v3:** award and links inside each project's chunk; training contexts built
by the same TF-IDF retriever the site uses; ~1,700 more training questions; dev F1 and
key-facts metrics; pretraining on WikiText-2 turned on.


In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "· CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Get the code (re-running this cell pulls the latest version)
import os
REPO = "https://github.com/Alfonso3435/portfolio.git"
if not os.path.exists("/content/portfolio"):
    !git clone -q {REPO} /content/portfolio
%cd /content/portfolio
!git pull -q
!git log -1 --format="Commit %h · %s"

In [ ]:
# 3. Phase 1 — export the site data and build the dataset (needs Node.js for the TypeScript export)
!node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y -qq nodejs)
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!npm run --silent export:knowledge
!python ml/build_dataset.py --samples 0

In [ ]:
# 4. Phase 2 — train the tokenizer (deterministic: identical to the one on your laptop)
!python ml/train_tokenizer.py | tail -8

### Pretraining on general English (stage A)

**On in v3.** Before learning the Q&A task, the model does plain next-token prediction on
WikiText-2 (~11 MB of Wikipedia articles, downloaded from Hugging Face) for 3,000 steps, so it
learns general English first. To measure whether it helps, set `PRETRAIN = False` and run again
under another `RUN` name — a good experiment for the Lab page.


In [ ]:
# 5. Download pretraining text
PRETRAIN = True  # ← False skips stage A
PRETRAIN_ARGS = ""
if PRETRAIN:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset
    wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    with open("ml/data/wikitext2.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(t for t in wiki["text"] if t.strip()))
    PRETRAIN_ARGS = "--pretrain-text ml/data/wikitext2.txt --pretrain-steps 3000"
    print("Pretraining text ready.")


In [ ]:
# 6. Train. Every 4 epochs it prints dev EM (exactly right), F1 (word overlap) and facts
#    (all names/numbers of the reference present). Early stopping keeps the best checkpoint.
RUN = "colab-v3"
!python ml/train.py --run {RUN} {PRETRAIN_ARGS}


In [ ]:
# 7. Curves — saved as a PNG for the Lab page
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"ml/checkpoints/{RUN}/metrics.jsonl")]
ft = [r for r in rows if r["stage"] == "finetune"]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot([r["epoch"] for r in ft], [r["train_loss"] for r in ft], label="train")
a.plot([r["epoch"] for r in ft], [r["dev_loss"] for r in ft], label="dev")
a.set(title="Loss (answer tokens)", xlabel="epoch", ylabel="cross-entropy"); a.legend()
ev = [r for r in ft if "dev_em" in r]
for key, label in [("dev_em", "exact match"), ("dev_f1", "token F1"), ("dev_facts", "key facts")]:
    b.plot([r["epoch"] for r in ev], [100 * r.get(key, 0) for r in ev], marker="o", label=label)
b.set(title="Dev answers", xlabel="epoch", ylabel="%", ylim=(0, 100)); b.legend()
plt.tight_layout(); plt.savefig(f"ml/checkpoints/{RUN}/curves.png", dpi=150); plt.show()


In [ ]:
# 8. Full pipeline on dev (retrieval + model + guardrails), the Phase 5 table.
#    Per-question details go into the run folder for error analysis.
!python ml/eval_pipeline.py --checkpoint ml/checkpoints/{RUN}/best.pt --device cuda --thresholds 0.10 0.05 0.20 --out ml/checkpoints/{RUN}/pipeline_dev.jsonl


In [ ]:
# 9. Ask it anything — the same pipeline the site will use
!python ml/ask.py --checkpoint ml/checkpoints/{RUN}/best.pt --device cuda --debug "What did he build at HackMX?" "Did SkySAR win anything?" "Does he know Keras?" "Has he used React Native?" "What's the capital of France?"


In [ ]:
# 10. Download the trained model, metrics and evaluation to your computer
!cd ml/checkpoints && zip -qr /content/{RUN}.zip {RUN}
from google.colab import files
files.download(f"/content/{RUN}.zip")
